In [5]:
#!pip -q install -U bertopic  hdbscan  plotly


In [1]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

train_df = pd.read_csv("preprocessed_train_final.csv")


c:\Users\ilker\anaconda3\envs\torchgpu\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def stratified_sample(
    df: pd.DataFrame,
    label_col: str,
    n_total: int,
    min_per_class: int = 5,
    seed: int = 42,
):
    # counts per class
    counts = df[label_col].value_counts()
    total = counts.sum()

    # proportional allocation
    take = (counts / total * n_total).round().astype(int)

    # ensure at least min_per_class (but not more than available)
    take = take.clip(lower=min_per_class, upper=counts)

    # adjust to hit n_total exactly (optional but nice)
    diff = int(n_total - take.sum())
    if diff != 0:
        # add/remove from largest classes (that still have room)
        order = counts.index.tolist()
        if diff > 0:
            for c in order:
                if diff == 0: break
                if take[c] < counts[c]:
                    take[c] += 1
                    diff -= 1
        else:
            for c in order:
                if diff == 0: break
                if take[c] > min_per_class:
                    take[c] -= 1
                    diff += 1

    # random rank per row inside each class using a single global shuffle key
    rng = np.random.default_rng(seed)
    u = rng.random(len(df))

    tmp = df[[label_col]].copy()
    tmp["_u"] = u

    # sort by (class, random) then take top N_i per class via cumcount
    sorted_idx = tmp.sort_values([label_col, "_u"]).index
    tmp2 = tmp.loc[sorted_idx]
    tmp2["_rn"] = tmp2.groupby(label_col).cumcount()

    take_df = take.rename("_take").to_frame()
    tmp2 = tmp2.join(take_df, on=label_col)

    sampled_idx = tmp2.index[tmp2["_rn"] < tmp2["_take"]]
    return df.loc[sampled_idx].copy()

# usage
# sample_df = stratified_sample(full_df, label_col="category", n_total=200_000, min_per_class=10)


train_df["text_len"] = train_df["product_text"].str.len().fillna(0)

# 10 length buckets (quantiles)
train_df["len_bin"] = pd.qcut(train_df["text_len"], q=10, duplicates="drop")

# combined strata (category + length bucket)
train_df["strata"] = train_df["category"].astype(str) + "||" + train_df["len_bin"].astype(str)

train_df = stratified_sample(train_df, label_col="strata", n_total=100_000, min_per_class=12, seed=42)

# keep original columns, drop helper cols if you want
train_df = train_df.drop(columns=["text_len", "len_bin", "strata"])

In [3]:
import json
category_mapping = json.load(open('category_map.json', 'r', encoding='utf-8'))


In [4]:
import re
import pandas as pd

TEXT_COL = "text_for_model"

dfT = train_df[["product_id", TEXT_COL]].copy()
dfT[TEXT_COL] = dfT[TEXT_COL].fillna("").astype(str)

def clean_for_topics(s: pd.Series) -> pd.Series:
    s = s.str.lower()
    # replace punctuation with space
    s = s.str.replace(r"[^\w\sğüşöçıİĞÜŞÖÇ]", " ", regex=True)
    # remove numeric+unit patterns
    s = s.str.replace(r"\b\d+([.,]\d+)?\s*(mm|cm|m|ml|l|lt|gr|g|kg|mg)\b", " ", regex=True)
    s = s.str.replace(r"\b\d+([.,]\d+)?(mm|cm|m|ml|l|lt|gr|g|kg|mg)\b", " ", regex=True)
    # remove packaging tokens
    s = s.str.replace(r"\b\d+\s*(adet|li|lü|lu|lı)\b", " ", regex=True)
    s = s.str.replace(r"\bx\s*\d+\b", " ", regex=True)
    # collapse spaces
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    return s

dfT["topic_text"] = clean_for_topics(dfT[TEXT_COL])

# junk filter (light)
keep = (dfT["topic_text"].str.len() >= 5) & dfT["topic_text"].str.contains(r"[a-zA-ZğüşöçıİĞÜŞÖÇ]", regex=True)
dfT = dfT.loc[keep].reset_index(drop=True)

dfT[["product_id","topic_text"]].head()


,product_id,topic_text
0,68716885,sfp port 1000base t module
1,68054762,hp aruba mm sr module
2,4018479,csh500p 5 port poe switch
3,2734243,wl889 300mbps 4 port router
4,2800552,gs 108s 8 port gigabit switch


In [5]:
import numpy as np

SEED = 42
SAMPLE_N = min(500_000, len(dfT))   # tune 200k–500k
dfS = dfT.sample(n=SAMPLE_N, random_state=SEED).reset_index(drop=True)

docs = dfS["topic_text"].tolist()
len(docs), docs[0][:120]


(169006, 'comfort tekerlekli temizlik seti')

In [6]:
from bertopic import BERTopic
from sentence_transformers import SentenceTransformer
import umap
import hdbscan

embedding_model_name = "Trendyol/TY-ecomm-embed-multilingual-base-v1.2.0"  # try -small if slow
embedder = SentenceTransformer(embedding_model_name, device="cuda", trust_remote_code=True)

umap_model = umap.UMAP(
    n_neighbors=30,
    n_components=5,
    min_dist=0.1,
    metric="cosine",
    random_state=SEED
)

hdbscan_model = hdbscan.HDBSCAN(
    min_cluster_size=200,          # tune: higher -> fewer topics, more stable
    min_samples=20,
    metric="euclidean",            # on UMAP space
    cluster_selection_method="eom",
    prediction_data=True
)

topic_model = BERTopic(
    embedding_model=embedder,
    umap_model=umap_model,
    hdbscan_model=hdbscan_model,
    calculate_probabilities=True,
    verbose=True
)


In [8]:
topics, probs = topic_model.fit_transform(docs)
topic_model.get_topic_info().head(10)

2026-02-10 20:53:45,733 - BERTopic - Embedding - Transforming documents to embeddings.
Batches: 100%|██████████| 5282/5282 [12:47<00:00,  6.88it/s]
2026-02-10 21:06:35,556 - BERTopic - Embedding - Completed ✓
2026-02-10 21:06:35,556 - BERTopic - Dimensionality - Fitting the dimensionality reduction algorithm
2026-02-10 21:11:17,437 - BERTopic - Dimensionality - Completed ✓
2026-02-10 21:11:17,441 - BERTopic - Cluster - Start clustering the reduced embeddings
2026-02-10 21:17:08,937 - BERTopic - Cluster - Completed ✓
2026-02-10 21:17:08,959 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-10 21:17:10,911 - BERTopic - Representation - Completed ✓


,Topic,Count,Name,Representation,Representative_Docs
0,-1,33315,-1_yağı_oyun_doğal_ve,"[yağı, oyun, doğal, ve, ahşap, boy, fidanı, se...","[hindistan cevizi yağı soğuk sıkım, plastik 4 ..."
1,0,3040,0_temizlik_bulaşık_temizleyici_deterjanı,"[temizlik, bulaşık, temizleyici, deterjanı, mo...",[konsantre sıvı bulaşık deterjanı ultra 5 litr...
2,1,2031,1_kalem_versatil_kalemi_tükenmez,"[kalem, versatil, kalemi, tükenmez, pastel, fa...",[marka grip 1345 0 5 versatil mavi renk uçlu k...
3,2,1533,2_nargile_ocak_mangal_çakmak,"[nargile, ocak, mangal, çakmak, şömine, ankast...","[fırınlı kuzine şömine camlı döküm soba siyah,..."
4,3,1513,3_ezmesi_glutensiz_katkısız_peyniri,"[ezmesi, glutensiz, katkısız, peyniri, fıstık,...","[100 saf katkısız fıstık ezmesi x, fıstık ezme..."
5,4,1239,4_rüzgarlığı_kapı_çamurluk_venti,"[rüzgarlığı, kapı, çamurluk, venti, kaput, kro...","[rover 75 çamurluk venti ve ayna rüzgarlığı, v..."
6,5,1186,5_çantası_omuz_çanta_postacı,"[çantası, omuz, çanta, postacı, portföy, clutc...",[kadın taba 5 bölmeli omuz çantası çapraz çant...
7,6,1109,6_eşofman_takım_takımı_kapüşonlu,"[eşofman, takım, takımı, kapüşonlu, ikili, yel...","[kapüşonlu yaka detaylı eşofman takım, kadın y..."
8,7,1105,7_led_aplik_ampul_feneri,"[led, aplik, ampul, feneri, aydınlatma, işık, ...",[bahçe aydınlatma set üstü fener aplik bahçe t...
9,8,1027,8_arabası_puset_kucağı_bebek,"[arabası, puset, kucağı, bebek, travel, ana, s...",[comfort siyah antrasit travel sistem bebek ar...


In [9]:
topic_model_5 = topic_model.reduce_topics(docs, nr_topics=5)
topic_model_5.get_topic_info()


2026-02-10 21:17:13,636 - BERTopic - Topic reduction - Reducing number of topics
2026-02-10 21:17:13,751 - BERTopic - Representation - Fine-tuning topics using representation models.
2026-02-10 21:17:15,228 - BERTopic - Representation - Completed ✓
2026-02-10 21:17:15,243 - BERTopic - Topic reduction - Reduced number of topics from 294 to 5


,Topic,Count,Name,Representation,Representative_Docs
0,-1,33315,-1_ve_siyah_seti_kadın,"[ve, siyah, seti, kadın, beyaz, ahşap, set, kı...","[kedi ve köpek için ağız ve diş bakım ürünü, ş..."
1,0,49784,0_seti_beyaz_bebek_takımı,"[seti, beyaz, bebek, takımı, ve, kişilik, siya...",[100 doğal pamuk nevresim seti tek kişilik lov...
2,1,35452,1_kadın_erkek_siyah_detaylı,"[kadın, erkek, siyah, detaylı, beden, beyaz, t...","[erkek siyah desenli boxer, kadın siyah kapüşo..."
3,2,26168,2_uyumlu_araç_şarj_siyah,"[uyumlu, araç, şarj, siyah, oto, telefon, ekra...",[huawei p smart pro uyumlu fm transmitter oto ...
4,3,24287,3_ve_kedi_saç_köpek,"[ve, kedi, saç, köpek, kokusu, bakım, paket, t...",[terebentin tuzsuz şampuan hindistan cevizi ya...


In [10]:
from tqdm.auto import tqdm
import numpy as np

all_docs = dfT["topic_text"].tolist()
N = len(all_docs)

B = 40_000
all_topics = np.empty(N, dtype=np.int32)
all_conf = np.empty(N, dtype=np.float32)

for start in tqdm(range(0, N, B), desc="Assign topics"):
    end = min(start+B, N)
    t, p = topic_model_5.transform(all_docs[start:end])
    all_topics[start:end] = np.array(t, dtype=np.int32)
    # confidence: max prob if available, else 1.0
    if p is None:
        all_conf[start:end] = 1.0
    else:
        all_conf[start:end] = p.max(axis=1).astype(np.float32)

dfT["topic_id"] = all_topics
dfT["topic_conf"] = all_conf
dfT["topic_id"].value_counts().head(20)


Batches: 100%|██████████| 1250/1250 [01:00<00:00, 20.71it/s]
2026-02-10 21:18:16,469 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-02-10 21:18:50,489 - BERTopic - Dimensionality - Completed ✓
2026-02-10 21:18:50,490 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-02-10 21:18:55,177 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-02-10 21:20:39,957 - BERTopic - Probabilities - Completed ✓
2026-02-10 21:20:39,957 - BERTopic - Cluster - Completed ✓
Batches: 100%|██████████| 1250/1250 [00:58<00:00, 21.33it/s]]
2026-02-10 21:21:38,981 - BERTopic - Dimensionality - Reducing dimensionality of input embeddings.
2026-02-10 21:21:52,191 - BERTopic - Dimensionality - Completed ✓
2026-02-10 21:21:52,192 - BERTopic - Clustering - Approximating new points with `hdbscan_model`
2026-02-10 21:21:57,113 - BERTopic - Probabilities - Start calculation of probabilities with HDBSCAN
2026-02-10 21:23:43,213 - BER

topic_id
 0    49121
-1    35348
 1    34616
 2    25785
 3    24136
Name: count, dtype: int64

In [11]:
largest_topic = dfT.loc[dfT["topic_id"] != -1, "topic_id"].value_counts().idxmax()
dfT["topic_id_fixed"] = dfT["topic_id"].where(dfT["topic_id"] != -1, largest_topic)

out = (dfT.groupby("topic_id_fixed")["product_id"]
         .apply(list)
         .reset_index(name="product_ids"))

out["topic_keywords"] = out["topic_id_fixed"].map(
    lambda t: ", ".join([w for w,_ in (topic_model_5.get_topic(t) or [])][:10])
)

out


,topic_id_fixed,product_ids,topic_keywords
0,0,"[46550917, 4580110, 130334199, 68373578, 68966...","seti, beyaz, bebek, takımı, ve, kişilik, siyah..."
1,1,"[104174463, 220175260, 201130694, 103971663, 1...","kadın, erkek, siyah, detaylı, beden, beyaz, ta..."
2,2,"[68716885, 68054762, 4018479, 2734243, 2800552...","uyumlu, araç, şarj, siyah, oto, telefon, ekran..."
3,3,"[50769039, 172589992, 147056173, 41865901, 755...","ve, kedi, saç, köpek, kokusu, bakım, paket, te..."


In [12]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF
import numpy as np

docs = dfS["topic_text"].tolist()  # sample to fit

tfidf = TfidfVectorizer(min_df=5, max_df=0.6, ngram_range=(1,2), max_features=200_000)
X = tfidf.fit_transform(docs)

nmf = NMF(n_components=5, random_state=SEED, init="nndsvda", max_iter=300)
W = nmf.fit_transform(X)   # doc-topic
H = nmf.components_        # topic-term

vocab = np.array(tfidf.get_feature_names_out())
for k in range(5):
    top = np.argsort(-H[k])[:15]
    print(f"\nTopic {k}:", ", ".join(vocab[top]))



Topic 0: kadın, siyah, kadın siyah, çantası, deri, takım, detaylı, beyaz, sandalet, ayakkabı, sütyen, omuz, siyah kadın, kadın beyaz, desenli

Topic 1: seti, ve, beyaz, set, bebek, boya, ahşap, duvar, mavi, boya seti, renk, çocuk, uyumlu, metal, desenli

Topic 2: takımı, kişilik, çift, nevresim, çift kişilik, nevresim takımı, tek, tek kişilik, kişilik nevresim, yatak, örtüsü, yatak örtüsü, pike, pamuk, saten

Topic 3: erkek, kol, saati, deri, kol saati, erkek siyah, lacivert, gri, erkek kol, siyah, ayakkabı, eşofman, yaka, boxer, hakiki

Topic 4: büyük, beden, büyük beden, boy, büyük boy, kadın büyük, eşofman, yaka, uzun, elbise, gömlek, tunik, lacivert, detaylı, cepli


In [13]:
def assign_nmf_topics(texts, batch=50_000):
    topic_id = np.empty(len(texts), dtype=np.int32)
    conf = np.empty(len(texts), dtype=np.float32)
    for start in tqdm(range(0, len(texts), batch), desc="NMF assign"):
        end = min(start+batch, len(texts))
        Xb = tfidf.transform(texts[start:end])
        Wb = nmf.transform(Xb)
        topic_id[start:end] = Wb.argmax(axis=1).astype(np.int32)
        conf[start:end] = (Wb.max(axis=1) / (Wb.sum(axis=1)+1e-12)).astype(np.float32)
    return topic_id, conf

all_topic_id, all_conf = assign_nmf_topics(dfT["topic_text"].tolist(), batch=50_000)
dfT["nmf_topic_id"] = all_topic_id
dfT["nmf_conf"] = all_conf


NMF assign: 100%|██████████| 4/4 [00:02<00:00,  1.60it/s]


In [14]:
dfT

,product_id,text_for_model,topic_text,topic_id,topic_conf,topic_id_fixed,nmf_topic_id,nmf_conf
0,68716885,sfp port 1000base t module,sfp port 1000base t module,2,1.000000,2,1,0.623961
1,68054762,hp aruba 10g mm sr module,hp aruba mm sr module,2,0.356144,2,1,0.825518
2,4018479,csh500p 5 port poe switch,csh500p 5 port poe switch,2,1.000000,2,1,0.675630
3,2734243,wl889 300mbps 4 port router,wl889 300mbps 4 port router,2,1.000000,2,1,0.626027
4,2800552,gs 108s 8 port gigabit switch,gs 108s 8 port gigabit switch,2,1.000000,2,1,0.669578
...,...,...,...,...,...,...,...,...
169001,117993923,artflame orta şömine 2 odunlu orta sehpa dağ e...,artflame orta şömine 2 odunlu orta sehpa dağ e...,0,1.000000,0,1,0.609827
169002,211197580,u tipi üç tarafı tam açık asansörlü şömine haz...,u tipi üç tarafı tam açık asansörlü şömine haz...,0,0.833057,0,1,0.517240
169003,70096089,60 cm ethanol burner bacasız şömine yanma hazn...,ethanol burner bacasız şömine yanma haznesi bi...,0,1.000000,0,1,0.572736
169004,78982715,nova şömine 90 lık design burner bioethanol ya...,nova şömine ık design burner bioethanol yakıt ...,0,0.823189,0,1,0.722618


In [27]:
dfT[dfT["topic_id"] == -1]

,product_id,text_for_model,topic_text,topic_id,topic_conf,topic_id_fixed,nmf_topic_id,nmf_conf,topic_title
11,46550917,m 100 4k wiress display dongle,m 100 4k wiress display dongle,-1,0.092029,0,2,0.367868,araç / şarj / siyah
21,4580110,71110 internet kontrol istasyonu,71110 internet kontrol istasyonu,-1,0.062332,0,1,1.000000,kadin / erkek / siyah
22,130334199,usb type c çok portlu yuva 8 port,usb type c çok portlu yuva 8 port,-1,0.097197,0,1,0.898447,kadin / erkek / siyah
61,68966360,digitus usb 2 0 serial ata iı sata iı adaptörün,digitus usb 2 0 serial ata iı sata iı adaptörün,-1,0.142549,0,1,0.613016,kadin / erkek / siyah
95,45996725,ubnt 4x4 mu mımo wave 2 802 11ac enterprıse wı...,ubnt 4x4 mu mımo wave 2 802 11ac enterprıse wı...,-1,0.178639,0,1,0.655341,kadin / erkek / siyah
...,...,...,...,...,...,...,...,...,...
168812,27707209,lisanslı grimelanj erkek çocuk bermuda takımı,lisanslı grimelanj erkek çocuk bermuda takımı,-1,0.187037,0,3,0.682530,kedi / saç / köpek
168817,36985550,tropical trip krem turkuaz erkek çocuk bermuda...,tropical trip krem turkuaz erkek çocuk bermuda...,-1,0.486008,0,3,0.896873,kedi / saç / köpek
168855,123319089,corvın ryker wash,corvın ryker wash,-1,0.131314,0,3,0.464595,kedi / saç / köpek
168905,78911347,büyük boy amigos taş chimena gri,büyük boy amigos taş chimena gri,-1,0.279438,0,4,0.926615,Topic 4


In [25]:
dfT.topic_id.value_counts()

topic_id
 0    49121
-1    35348
 1    34616
 2    25785
 3    24136
Name: count, dtype: int64

In [26]:
import re
import numpy as np
import pandas as pd

DOMAIN_STOP = set("""
cm mm ml l lt gr g kg mg adet pcs piece paket pack x x2 x3 x4 x5 li lu lü
orjinal orijinal uygun uyumlu set takim takım model seri serisi yeni
""".split())

def clean_kw(w: str):
    if w is None:
        return None
    w = str(w).strip().lower()
    w = w.replace("ı", "i").replace("İ", "i")
    w = re.sub(r"[^\w\sğüşöçıİĞÜŞÖÇ]+", " ", w)
    w = re.sub(r"\s+", " ", w).strip()

    if (not w) or (len(w) <= 2) or w.isdigit():
        return None
    if w in DOMAIN_STOP:
        return None
    # numeric + unit patterns
    if re.fullmatch(r"\d+([.,]\d+)?\s*(mm|cm|m|ml|l|lt|gr|g|kg|mg)\b", w):
        return None
    if re.fullmatch(r"\d+([.,]\d+)?(mm|cm|m|ml|l|lt|gr|g|kg|mg)\b", w):
        return None
    return w


In [28]:
# Pick which topic id column to name
TOPIC_COL = "topic_id"   # in your notebook this is the "fixed" topic id after handling -1
TEXT_COL  = "topic_text"

def build_bertopic_names(df, topic_model, topic_col=TOPIC_COL, topn=10, title_k=3, n_examples=3):
    rows = []
    for tid, g in df.groupby(topic_col):
        # keywords from BERTopic
        if int(tid) == -1:
            raw = []
        else:
            raw = topic_model.get_topic(int(tid)) or []

        kws = []
        for w, _ in raw[:topn*3]:
            cw = clean_kw(w)
            if cw and cw not in kws:
                kws.append(cw)
            if len(kws) >= topn:
                break

        title = " / ".join(kws[:title_k]) if kws else f"Topic {tid}"

        examples = g[TEXT_COL].astype(str).head(n_examples).tolist()

        rows.append({
            topic_col: int(tid),
            "topic_size": int(len(g)),
            "topic_title_auto": title,
            "topic_keywords_clean": kws,
            "examples": examples
        })

    names = pd.DataFrame(rows).sort_values("topic_size", ascending=False).reset_index(drop=True)
    return names

topic_names = build_bertopic_names(dfT, topic_model_5)
display(topic_names)


,topic_id,topic_size,topic_title_auto,topic_keywords_clean,examples
0,0,49121,seti / beyaz / bebek,"[seti, beyaz, bebek, takimi, kişilik, siyah, ç...",[1u 16a 10port basic pdu with surge protection...
1,-1,35348,Topic -1,[],"[m 100 4k wiress display dongle, 71110 interne..."
2,1,34616,kadin / erkek / siyah,"[kadin, erkek, siyah, detayli, beden, beyaz, t...","[kadın balık kesim taşlı abiye, kırmızı yırtma..."
3,2,25785,araç / şarj / siyah,"[araç, şarj, siyah, oto, telefon, ekran, kilif...","[sfp port 1000base t module, hp aruba mm sr mo..."
4,3,24136,kedi / saç / köpek,"[kedi, saç, köpek, kokusu, bakim, temizleme, k...","[pv7000 c polisaj makinası, otomatik kızak yağ..."


In [29]:
# Optional: override some names manually for presentation
manual_topic_names = {
    # 0: "Ev & Yaşam — Dekor / Aydınlatma",
    # 1: "Kozmetik — Makyaj / Bakım",
}

topic_title_map = dict(zip(topic_names[TOPIC_COL], topic_names["topic_title_auto"]))
topic_title_map.update(manual_topic_names)

dfT["topic_title"] = dfT[TOPIC_COL].map(topic_title_map).fillna("Unknown")
dfT[[TOPIC_COL, "topic_title"]].drop_duplicates().sort_values(TOPIC_COL).head(20)


,topic_id,topic_title
11,-1,Topic -1
54,0,seti / beyaz / bebek
324,1,kadin / erkek / siyah
0,2,araç / şarj / siyah
709,3,kedi / saç / köpek


In [30]:
EVAL_TOPIC_COL = "topic_id"
CONF_COL = "topic_conf"  # created in your notebook

# Basic size table
size_tbl = dfT[EVAL_TOPIC_COL].value_counts().rename("n").reset_index().rename(columns={"index": EVAL_TOPIC_COL})
size_tbl["pct"] = (size_tbl["n"] / len(dfT) * 100).round(2)
display(size_tbl)

# Confidence stats (if present)
if CONF_COL in dfT.columns:
    conf_stats = dfT.groupby(EVAL_TOPIC_COL)[CONF_COL].agg(
        n="size",
        mean_conf="mean",
        med_conf="median",
        p10=lambda x: float(np.quantile(x, 0.10)),
        p90=lambda x: float(np.quantile(x, 0.90)),
    ).reset_index()

    conf_stats["mean_conf"] = conf_stats["mean_conf"].round(4)
    conf_stats["med_conf"]  = conf_stats["med_conf"].round(4)
    conf_stats["p10"]       = conf_stats["p10"].round(4)
    conf_stats["p90"]       = conf_stats["p90"].round(4)

    display(conf_stats.sort_values("n", ascending=False))

    # Ambiguity rate = bottom 10% confidence (global)
    thr = float(np.quantile(dfT[CONF_COL].values, 0.10))
    amb_rate = float((dfT[CONF_COL].values <= thr).mean() * 100)
    print(f"Ambiguous rate (bottom 10% {CONF_COL}): {amb_rate:.1f}% | threshold: {thr:.6f}")
else:
    print(f"'{CONF_COL}' not found; ambiguity rate skipped.")


,topic_id,n,pct
0,0,49121,29.06
1,-1,35348,20.92
2,1,34616,20.48
3,2,25785,15.26
4,3,24136,14.28


,topic_id,n,mean_conf,med_conf,p10,p90
1,0,49121,0.7457,1.0000,0.1682,1.0000
0,-1,35348,0.2621,0.2221,0.0339,0.5504
2,1,34616,0.6965,0.9400,0.1169,1.0000
3,2,25785,0.6758,0.9568,0.0992,1.0000
4,3,24136,0.7531,0.9995,0.1404,1.0000


Ambiguous rate (bottom 10% topic_conf): 10.0% | threshold: 0.097818


In [31]:
from sklearn.metrics import silhouette_score

# Evaluate silhouette on a sample to keep it reasonable
SAMPLE_SIL = 50_000
seed = 42

# Exclude outliers if they still exist (shouldn't after topic_id_fixed, but safe)
labels = dfT[EVAL_TOPIC_COL].values
mask_valid = labels != -1

idx_all = np.where(mask_valid)[0]
n = len(idx_all)
n_sample = min(SAMPLE_SIL, n)

rs = np.random.RandomState(seed)
sample_idx = rs.choice(idx_all, size=n_sample, replace=False)

texts = dfT.iloc[sample_idx][TEXT_COL].astype(str).tolist()
y = dfT.iloc[sample_idx][EVAL_TOPIC_COL].values

# embed on GPU (batch size tune if needed)
emb = embedder.encode(
    texts,
    batch_size=512,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True
).astype(np.float32)

# need at least 2 clusters in sample
if len(np.unique(y)) >= 2:
    sil = silhouette_score(emb, y, metric="cosine")
    print("Silhouette (cosine, sample):", float(sil))
else:
    print("Not enough distinct clusters in sample for silhouette.")


Batches: 100%|██████████| 98/98 [01:25<00:00,  1.15it/s]


Silhouette (cosine, sample): 0.037356436252593994


In [32]:
NMF_TOPIC_COL = "topic_id"
NMF_CONF_COL  = "nmf_conf"

# ---- Name NMF topics using top TF-IDF terms from nmf.components_ (already in your notebook) ----
# We reuse vocab from your notebook: vocab = np.array(tfidf.get_feature_names_out())
# and H = nmf.components_

nmf_names = []
topn = 12

for k in range(nmf.n_components):
    top_idx = np.argsort(-H[k])[:topn*3]
    kws = []
    for j in top_idx:
        cw = clean_kw(vocab[j])
        if cw and cw not in kws:
            kws.append(cw)
        if len(kws) >= topn:
            break
    title = " / ".join(kws[:3]) if kws else f"NMF {k}"
    nmf_names.append((k, title, kws))

nmf_names_df = pd.DataFrame(nmf_names, columns=[NMF_TOPIC_COL, "nmf_title_auto", "nmf_keywords_clean"])
display(nmf_names_df)

nmf_title_map = dict(zip(nmf_names_df[NMF_TOPIC_COL], nmf_names_df["nmf_title_auto"]))
dfT["nmf_title"] = dfT[NMF_TOPIC_COL].map(nmf_title_map).fillna("Unknown")

# ---- Evaluate NMF clusters ----
nmf_size = dfT[NMF_TOPIC_COL].value_counts().rename("n").reset_index().rename(columns={"index": NMF_TOPIC_COL})
nmf_size["pct"] = (nmf_size["n"] / len(dfT) * 100).round(2)
display(nmf_size)

if NMF_CONF_COL in dfT.columns:
    thr = float(np.quantile(dfT[NMF_CONF_COL].values, 0.10))
    amb_rate = float((dfT[NMF_CONF_COL].values <= thr).mean() * 100)
    print(f"NMF ambiguous rate (bottom 10% {NMF_CONF_COL}): {amb_rate:.1f}% | threshold: {thr:.6f}")


,topic_id,nmf_title_auto,nmf_keywords_clean
0,0,kadin / siyah / kadin siyah,"[kadin, siyah, kadin siyah, çantasi, deri, det..."
1,1,seti / beyaz / bebek,"[seti, beyaz, bebek, boya, ahşap, duvar, mavi,..."
2,2,takimi / kişilik / çift,"[takimi, kişilik, çift, nevresim, çift kişilik..."
3,3,erkek / kol / saati,"[erkek, kol, saati, deri, kol saati, erkek siy..."
4,4,büyük / beden / büyük beden,"[büyük, beden, büyük beden, boy, büyük boy, ka..."


,topic_id,n,pct
0,0,49121,29.06
1,-1,35348,20.92
2,1,34616,20.48
3,2,25785,15.26
4,3,24136,14.28


NMF ambiguous rate (bottom 10% nmf_conf): 10.0% | threshold: 0.501277
